|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>CUDA graphs<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: capture a step, then survive a changing batch<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import cudalib

Capture a decode step as a CUDA graph, then make it survive a batch size
that changes every step.

This is stage 12. The capture is twenty lines and three of them are traps.

In [2]:
### run this cell: a stand-in decode step

class Block(nn.Module):
  def __init__(self, d):
    super().__init__()
    self.n1, self.n2 = nn.RMSNorm(d), nn.RMSNorm(d)
    self.qkv = nn.Linear(d, 3*d, bias=False)
    self.o   = nn.Linear(d, d, bias=False)
    self.up  = nn.Linear(d, 4*d, bias=False)
    self.dn  = nn.Linear(4*d, d, bias=False)
  def forward(self, x):
    h = self.n1(x); q,k,v = self.qkv(h).chunk(3,-1)
    x = x + self.o(torch.softmax(q @ k.transpose(-1,-2)/32.0, -1) @ v)
    h = self.n2(x)
    return x + self.dn(F.silu(self.up(h)))

class Model(nn.Module):
  def __init__(self, n=28, d=1024):
    super().__init__(); self.blocks = nn.ModuleList([Block(d) for _ in range(n)])
  def forward(self, x):
    for b in self.blocks: x = b(x)
    return x

D = 1024
model = Model(28, D).cuda().to(torch.bfloat16).eval()
model.requires_grad_(False)
print('28 layers, hidden', D)

28 layers, hidden 1024


# Exercise 1: capture and replay

Warm up, record, replay. Measure what it bought.

In [3]:
def capture(model, example):
  static_in = example.clone()
  s = torch.cuda.Stream(); s.wait_stream(torch.cuda.current_stream())
  with torch.cuda.stream(s):
    for _ in range(3): model(static_in)
  torch.cuda.current_stream().wait_stream(s)
  g = torch.cuda.CUDAGraph()
  with torch.cuda.graph(g):
    static_out = model(static_in)
  return g, static_in, static_out

with torch.no_grad():
  x = torch.randn(1, 1, D, device='cuda', dtype=torch.bfloat16)
  g, si, so = capture(model, x)
  eager = cudalib.bench_ms(lambda: model(x), iters=50, warmup=20, best_of=3)
graph = cudalib.bench_ms(lambda: g.replay(), iters=50, warmup=20, best_of=3)
print(f'eager {eager:.3f} ms, graph {graph:.3f} ms  ({eager/graph:.2f}x)')

eager 5.293 ms, graph 3.111 ms  (1.70x)


# Exercise 2: feeding it

Replay runs exactly what was recorded, reading and writing exactly the
buffers it recorded. Get your data into those buffers.

In [4]:
real = torch.randn(1, 1, D, device='cuda', dtype=torch.bfloat16)

si.copy_(real)            # in place. assigning a new tensor is not seen.
g.replay()
from_graph = so.clone()

with torch.no_grad():
  from_eager = model(real)
print('agree:', torch.allclose(from_graph, from_eager, rtol=1e-2, atol=1e-2))

# and the trap: a fresh tensor is invisible to the graph
si = torch.randn(1, 1, D, device='cuda', dtype=torch.bfloat16)   # rebinding!
g.replay()
print('still the OLD output:', torch.allclose(so, from_graph))

agree: True
still the OLD output: True


# Exercise 3: shape buckets

A server's batch size changes every step and a graph's does not. Capture
several and pad up to the nearest.

In [5]:
BUCKETS = [1, 2, 4, 8, 16, 32]
graphs = {}
with torch.no_grad():
  for b in BUCKETS:
    graphs[b] = capture(model, torch.randn(b, 1, D, device='cuda', dtype=torch.bfloat16))

def bucket_for(n):
  return next((b for b in BUCKETS if b >= n), None)

def run(n):
  b = bucket_for(n)
  if b is None:
    with torch.no_grad(): return model(torch.randn(n,1,D,device='cuda',dtype=torch.bfloat16))
  gg, sin, sout = graphs[b]
  sin[:n].copy_(torch.randn(n,1,D,device='cuda',dtype=torch.bfloat16))
  gg.replay()
  return sout[:n]

print(f"{'seqs':>5} {'bucket':>7} {'padding':>8} {'eager ms':>10} {'graph ms':>10} {'gain':>6}")
for n in (1, 3, 5, 12, 31, 48):
  xn = torch.randn(n,1,D,device='cuda',dtype=torch.bfloat16)
  with torch.no_grad():
    e = cudalib.bench_ms(lambda: model(xn), iters=30, warmup=10, best_of=2)
  gm = cudalib.bench_ms(lambda: run(n), iters=30, warmup=10, best_of=2)
  b = bucket_for(n)
  pad = f'{100*(b-n)/b:.0f}%' if b else 'n/a'
  print(f'{n:>5} {str(b):>7} {pad:>8} {e:>10.3f} {gm:>10.3f} {e/gm:>5.2f}x')

 seqs  bucket  padding   eager ms   graph ms   gain


    1       1       0%      5.596      3.136  1.78x


    3       4      25%      5.897      3.163  1.86x


    5       8      38%      6.046      3.182  1.90x


   12      16      25%      4.672      3.380  1.38x


   31      32       3%      5.059      3.388  1.49x


   48    None      n/a      7.307      7.235  1.01x


# Exercise 4: what the buckets cost

Not time. Memory.

In [6]:
free_before = torch.cuda.mem_get_info()[0]
extra = {}
with torch.no_grad():
  for b in (64, 128):
    extra[b] = capture(model, torch.randn(b,1,D,device='cuda',dtype=torch.bfloat16))
free_after = torch.cuda.mem_get_info()[0]

cost = (free_before - free_after)/1e6
print(f'2 more graphs cost {cost:.0f} MB of VRAM')
print(f'at 112 KB per token of KV cache, that is '
      f'{cost*1e6/(112*1024):,.0f} tokens the pool does not get')

2 more graphs cost 55 MB of VRAM
at 112 KB per token of KV cache, that is 475 tokens the pool does not get


### Three things you now know that a tutorial would not tell you

**The warm-up on a side stream is load-bearing.** cuBLAS picks an
algorithm and allocates a workspace the first time it sees a shape.
Capture that and the graph records an allocation, which is at best a
leak and at worst a crash on replay.

**The graph holds pointers, not values.** Rebinding `si` to a new tensor
changes a Python name and nothing else; replay keeps reading the buffer
it recorded. That is why the second half of Exercise 2 produces stale
output and no error at all.

**Buckets cost memory, and the memory comes out of the KV pool.** Part 3
spent four sections getting that pool back. A generous bucket list
quietly spends some of it, and the only way to know how much is to
measure, as Exercise 4 does.

So the bucket list is not a free win to be maximised. It is a trade
against concurrency, and the right size for it depends on the batch
sizes your traffic actually produces.

    ./vc guide 12